In [1]:
import pandas as pd


# File paths and other parameters
Nbins= 2
property= 'Activity'
folder= 'Feature_Importance'
header_name= "Feature List Name"
metrics_file_path = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{folder}\{property}_Metrics {Nbins} bins.xlsx"
class_info_file_path = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_Class_Information_{Nbins} bins.xlsx"
output_file_path = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{folder}\{property}_Weighted_Average_Metrics {Nbins} bins.xlsx"


def compute_weighted_metrics(row, test_counts, n_classes):

    # Common metric name stems for each class:
    metric_stems = ["Accuracy", "Precision", "Recall", "F1 Score", "MCC", "AUC"]
    
    # Sums for denominator
    total_test = sum(test_counts)
    
    # Prepare dict of weighted metrics
    weighted_values = {}
    for metric_name in metric_stems:
        # Sum over all classes:  ( metric_i * test_count_i )
        num = 0.0
        for class_idx in range(n_classes):
            col = f"{metric_name}_{class_idx}"
            num += row[col] * test_counts[class_idx]
        weighted_values[f"{metric_name}"] = num / total_test
    
    # Copy Kappa (not class-based)
    weighted_values["Kappa"] = row["Kappa"]
    
    return weighted_values


def process_metrics(metrics_file, class_info_file, output_file, n_classes):

    folds = [f"Fold {i}" for i in range(1, 6)]
    
    # We will accumulate all Weighted-Fold dataframes in a list so we can average them
    weighted_folds_list = []
    
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        
        for fold_name in folds:
            # Read the metrics for this fold
            metrics_df = pd.read_excel(metrics_file, sheet_name=fold_name)
            
            # Read the class info for this fold
            #   We assume the columns are e.g. Train_Count_0, Test_Count_0, ...
            #   and that the data row is the first row of actual data (index=0 after reading).
            class_info_df = pd.read_excel(class_info_file, sheet_name=fold_name)
            
            # Extract training counts from the single row in class_info_df
            # For n_classes=2, we have columns: Train_Count_0, Test_Count_0, Train_Count_1, Test_Count_1
            # For n_classes=4, we have columns: Train_Count_0, Test_Count_0, Train_Count_1, Test_Count_1, ...
            test_counts = []
            for c_idx in range(n_classes):
                test_col = f"Test_Count_{c_idx}"
                test_counts.append(class_info_df.loc[0, test_col])
            
            # Build a new DataFrame to hold the weighted metrics
            weighted_df = pd.DataFrame()
            weighted_df[header_name] = metrics_df[header_name]
            
            # For each row in metrics_df, compute the weighted metrics
            weighted_rows = []
            for _, row in metrics_df.iterrows():
                wvals = compute_weighted_metrics(row, test_counts, Nbins)
                weighted_rows.append(wvals)
            
            weighted_rows_df = pd.DataFrame(weighted_rows)
            # Merge the (ML_Model, Featurizer) columns with the newly computed weighted columns
            weighted_df = pd.concat([weighted_df, weighted_rows_df], axis=1)
            
            # Write this Weighted Fold to the output Excel
            sheet_name = f"Weighted {fold_name}"
            weighted_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Keep track of which fold it came from (for later averaging)
            weighted_df["Fold"] = fold_name
            weighted_folds_list.append(weighted_df)
        
        # --------------------------------------------
        # After all folds are processed, compute average across folds
        # --------------------------------------------
        all_weighted = pd.concat(weighted_folds_list, ignore_index=True)
        
        # Group by (ML_Model, Featurizer) and compute mean of numeric columns
        # That will include Weighted_Accuracy, Weighted_Precision, etc., and Kappa
        group_cols = [header_name]
        avg_df = all_weighted.groupby(group_cols, sort=False).mean(numeric_only=True).reset_index()
        
        # The groupby mean() will include "Fold" as well, which isn't meaningful to average;
        # drop it if you want to keep your final table clean:
        if "Fold" in avg_df.columns:
            avg_df.drop(columns=["Fold"], inplace=True)
        
        # Write the final averaged metrics to 'Fold Avg'
        avg_df.to_excel(writer, sheet_name="Fold Avg", index=False)


process_metrics(metrics_file_path, class_info_file_path, output_file_path, Nbins)

print("Finished")





Finished
